# Training and Processing
  202511: removed 1799 outlier rows (9191 -> 7392)
Processed 202511: 7392 rows
  202512: removed 1986 outlier rows (9889 -> 7903)
Processed 202512: 7903 rows
  202601: removed 1481 outlier rows (7022 -> 5541)
Processed 202601: 5541 rows
  202602: removed 1589 outlier rows (8065 -> 6476)
Processed 202602: 6476 rows
  202603: removed 2075 outlier rows (10590 -> 8515)
Processed 202603: 8515 rows
  202604: removed 2170 outlier rows (11363 -> 9193)
Processed 202604: 9193 rows
  202605: removed 2191 outlier rows (11339 -> 9148)
Processed 202605: 9148 rows

Training on months: ['202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604']
Testing on month: 202605

# Linear Regression vs. Decision Tree vs. Random Forest
## === Baseline: Linear Regression ===
  train_size=98618  RMSE=$199,887  MAE=$136,339  R²=0.856
Saved to baseline_linear_regression_results.csv

## === Decision Tree ===
  train_size=98618  RMSE=$392,174  MAE=$283,764  R²=0.444

## === Random Forest ===
  train_size=98618  RMSE=$379,036  MAE=$279,245  R²=0.480

## === Model Comparison (test R²) ===
            model               rmse           mae            r2
          Linear Regression     199886.733828 136339.286040 0.855514
          Decision Tree         392173.931963 283763.576112 0.443821
          Random Forest         379036.243027 279244.637318 0.480460

# Top 20 feature importances (Random Forest):
LivingArea               0.408778
YearBuilt                0.222270
BathroomsTotalInteger    0.078617
LotSizeSquareFeet        0.055024
FireplaceYN              0.014332
GarageSpaces             0.013880
BedroomsTotal            0.006676
zip_95124                0.006162
zip_94539                0.005834
zip_92677                0.005807
zip_90266                0.004347
zip_92024                0.004300
zip_92694                0.004176
zip_92651                0.004118
zip_92618                0.003781
zip_95120                0.003759
zip_95051                0.003674
zip_92127                0.003602
zip_94582                0.003563
zip_95125                0.003514
dtype: float64

# Week 5 Model Comparison Notes
# ==============================
Test month: 202605
Train months: 202505, 202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604
Train size: 98618 rows

# Results (test set):
  Linear Regression -> RMSE=$199,887  MAE=$136,339  R²=0.856
  Decision Tree      -> RMSE=$392,174  MAE=$283,764  R²=0.444
  Random Forest      -> RMSE=$379,036  MAE=$279,245  R²=0.480

# Linear Regression
- Strengths: Fast to train, coefficients are directly interpretable
  (e.g. "$X per additional sqft" holding other features fixed), and it
  is a stable, low-variance baseline that's hard to overfit given
  enough rows relative to the number of zip-code dummy columns.
  
- Weaknesses: Assumes a linear, additive relationship between features
  and price. It can't capture interactions (e.g. an extra bedroom is
  worth more in some zip codes than others) or non-linear effects
  (e.g. diminishing returns on LivingArea past a certain size) unless
  those interactions are explicitly engineered as features.

# Decision Tree
- Strengths: Captures non-linear relationships and feature interactions
  automatically (e.g. it can learn "zip=X AND LivingArea>2500 ->
  premium" without being told to). Also interpretable via its splits,
  and needs no feature scaling.
  
- Weaknesses: A single tree is high-variance and prone to overfitting,
  especially with many one-hot zip columns providing lots of ways to
  split on location alone. Small changes in the training data can
  produce a very different tree, and predictions are piecewise-constant
  (it can only predict values seen in training leaves), which tends to
  hurt RMSE/MAE versus a smoother model.

# Random Forest
- Strengths: Averaging many decorrelated trees reduces the variance/
  overfitting problem of a single Decision Tree while keeping the
  ability to model non-linearities and interactions. Typically the best
  R² of the three here, and feature_importances_ gives a useful ranking
  of which fields (numeric features vs. specific zip dummies) drive
  price the most.
  
- Weaknesses: Slower to train and to run inference on than the other
  two models (300 trees vs. 1 tree vs. 1 linear model), less directly
  interpretable than a single tree or linear coefficients, and it still
  can't extrapolate outside the price/feature ranges seen in training
  (e.g. a brand-new luxury zip code with no sales history).

# Deliverable checklist:
  [x] Decision Tree and Random Forest regressors trained

  [x] Test R² compared against Linear Regression baseline
      (see model_comparison_results.csv)
      
  [x] Model behavior (strengths/weaknesses) documented above

Normally you'd expect Random Forest ≥ Decision Tree ≥ Linear Regression on a problem like this, since tree ensembles usually capture non-linearities and interactions that linear models miss. Seeing it reversed this hard is a real signal, not just noise — worth digging into rather than just reporting it.
The most likely culprit: one-hot encoded zip codes.
Your model_features list is numeric_features + zip_dummy_cols — and if you have, say, 100+ distinct zip codes in your market, that's 100+ sparse binary columns dominating the feature space.

Linear Regression handles this gracefully. It just assigns each zip dummy its own coefficient (a location-based price adjustment) and adds them up. That's exactly the right way to represent "location shifts price by roughly $X."
Trees handle this badly. A tree can only split on one zip dummy at a time ("is this zip==90210? yes/no"), so encoding location as 100 separate binary columns forces the tree to rediscover the same information through many fragmented splits instead of one clean categorical decision. It also means at each split the tree has ~100+ near-identical, low-signal columns to sift through, which invites overfitting to noise rather than finding the numeric-feature splits (bedrooms, sqft, etc.) that actually generalize.

That's consistent with what you're seeing: both tree models land in roughly the same degraded range (R² 0.44–0.48), while linear regression — which has no trouble with high-dimensional dummy variables — comes out well ahead.
Secondary contributor: max_depth=12 combined with the eaten-up split budget. With 100+ zip dummies plus 13 numeric features, a depth-12 tree is spending a lot of its splits threading through sparse dummies instead of building efficient decision boundaries on the numeric features.
What this means for your writeup / next steps, if you want to push R² higher on the tree models:

Try target/mean encoding the zip code instead of one-hot (replace each zip with, e.g., its historical average price) — trees do much better with a single continuous "location value" column than with dozens of dummies.
Or drop max_depth and tune min_samples_leaf/max_features to reduce overfitting from all those dummy splits.
Report this as a finding, not just a table — "tree-based models underperformed linear regression here, likely due to high-cardinality one-hot encoding of PostalCode diluting split quality" is a legitimate and useful conclusion for a Week 5 deliverable, especially paired with the feature importances CSV (check whether zip dummies are actually dominating the top of that list).

Want me to modify the script to add a mean-encoded zip version so you can compare both encoding strategies?Want to be notified when Claude responds?